# Playground S6E8 — Mask + Student（最终提交）

私榜更好的那份：`R3 + 9 列 mask + mask 分组 LR + 蒸馏 student`。

| | 5 折 OOF | 公榜 | 私榜 |
|---|---|---|---|
| mask_r3 | 0.970023 | 0.97099 | 较低 |
| **本 notebook（mask_student）** | **0.970158** | 0.97093 | **较高（最终成绩）** |

```text
StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
C = 1
```

只训一层元模型：logit 后的 L2 逻辑回归（Newton，`C * Σ logloss + ½‖w‖²`，截距不惩罚）。
标签只在折内 train 行上用。提交 = 五折 test 概率平均。

**169 列：** 135 公开 OOF logit + 19 extras + 4 来源分组 LR + 9 mask logit + 1 mask 分组 LR + 1 student logit。

公开成员来源（同折、CC0 / 公开 Dataset）：

- 74 库 `oof_lib/oof/`（szymonkapiski/s6e8-oof-library-47-models）
- 50 weak `oof_extra/weak50/`
- Paiky 仅 5 折 6 列 `oof_extra/paiky11/`（不用 `_10f`）
- 5 FM `oof_extra/fm/`
- 9 mask `oof_extra/mask/oof/`（aadijoshi19/s6e8-mask-augmented-oof-library）
- student `oof_extra/transductive/oof_soft_student.npy`（raykkretzschmar/s6e8-transductive-anti-student-signals）

不用 V13、不用 10 折、不用 naji blend。Run All 写出 `submission.csv`。


## 1. 路径与 Newton 逻辑回归

sklearn 的 `lbfgs` 在这个宽度上会崩，所以自己做 Newton，目标与 `LogisticRegression(C=1, penalty='l2')` 一致。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = Path(".").resolve()
DATA = ROOT / "data"
LIB = ROOT / "oof_lib" / "oof"
WEAK = ROOT / "oof_extra" / "weak50"
PAIKY = ROOT / "oof_extra" / "paiky11"
FM = ROOT / "oof_extra" / "fm"
MASK = ROOT / "oof_extra" / "mask" / "oof"
STU_OOF = ROOT / "oof_extra" / "transductive" / "oof_soft_student.npy"
STU_TE = ROOT / "oof_extra" / "transductive" / "test_soft_student.npy"
TARGET = "addicted_label"
N_FOLDS, SEED, C = 5, 42, 1.0
PAIKY_5F = ("v10_tabm", "v13_lookup", "v14_lookup_bag", "v15_lookup_wide", "v16_lookup_aug", "v17_realmlp")
FM_NAMES = ("fmplr", "fmnum", "fmdeep", "fmwide", "fmpure")
MASK_NAMES = (
    "latmask_xgb2", "latmask_xgb", "latmask_lgbm",
    "lmx_nolat", "lml_nolat", "lmx_latonly", "lml_latonly",
    "xgb_v1", "lml_rawonly",
)
# 公开 135 列的来源切片：74 库 | 50 weak | 6 Paiky | 5 FM
SLICES = {"lib74": (0, 74), "weak50": (74, 124), "paiky": (124, 130), "fm": (130, 135)}

assert (DATA / "train.csv").exists()
assert MASK.exists() and STU_OOF.exists()

def clip01(a):
    return np.clip(a, 1e-6, 1.0 - 1e-6)

def to_logit(P):
    return logit(clip01(P))

def _xtwx(X, w, C, chunk=65536):
    n, d = X.shape
    H = np.zeros((d, d), dtype=np.float64)
    for i in range(0, n, chunk):
        sl = slice(i, min(i + chunk, n))
        Xi = X[sl]
        H += (Xi * w[sl, None]).T @ Xi
    return H * C

def fit_logreg(X, y, C=C, max_iter=40):
    n, d = X.shape
    Xb = np.column_stack([X, np.ones(n, dtype=np.float64)])
    w = np.zeros(d + 1, dtype=np.float64)
    y = y.astype(np.float64)
    eye = np.eye(d + 1, dtype=np.float64)
    eye[-1, -1] = 0.0
    for _ in range(max_iter):
        z = np.clip(Xb @ w, -30.0, 30.0)
        p = expit(z)
        g = C * (Xb.T @ (p - y)) + (w * np.r_[np.ones(d), 0.0])
        ww = p * (1.0 - p)
        H = _xtwx(Xb, ww, C) + eye
        try:
            step = np.linalg.solve(H, g)
        except np.linalg.LinAlgError:
            step = np.linalg.lstsq(H, g, rcond=None)[0]
        w = w - step
        if float(np.max(np.abs(step))) < 1e-7:
            break
    return w

def predict_proba(X, w):
    return expit(np.clip(X @ w[:-1] + w[-1], -30.0, 30.0))

def stack_ready(X, y, Xt, tag):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y), dtype=np.float64)
    pred = np.zeros(len(Xt), dtype=np.float64)
    for i, (tr, va) in enumerate(skf.split(X, y), 1):
        w = fit_logreg(X[tr], y[tr], C=C)
        oof[va] = predict_proba(X[va], w)
        pred += predict_proba(Xt, w) / N_FOLDS
        print(f"  [{tag}] fold {i} {roc_auc_score(y[va], oof[va]):.6f}", flush=True)
    auc = float(roc_auc_score(y, oof))
    print(f"  [{tag}] OOF {auc:.6f}", flush=True)
    return oof, pred, auc

print("ok", ROOT)


## 2. 加载公开 135 + mask 9 + student，以及 extras / 折内分组 LR

extras（19 列）是行/来源/族统计，**不算标签**。来源分组 LR 和 mask 分组 LR 都在折内拟合。


In [ ]:
def load_public(n_train, n_test, train_ids):
    oof_c, te_c, names = [], [], []
    for op in sorted(LIB.glob("oof_*.npy")):
        name = op.name[len("oof_") : -4]
        tp = LIB / f"test_{name}.npy"
        if not tp.exists():
            continue
        o = np.load(op).astype(np.float64).reshape(-1)
        t = np.load(tp).astype(np.float64).reshape(-1)
        if o.shape[0] != n_train or t.shape[0] != n_test:
            continue
        if not np.isfinite(o).all() or not np.isfinite(t).all():
            continue
        names.append(name)
        oof_c.append(o)
        te_c.append(t)
    Xw = np.load(WEAK / "oof.npy").astype(np.float64)
    Tw = np.load(WEAK / "test.npy").astype(np.float64)
    po = pd.read_csv(PAIKY / "oof_predictions.csv")
    pt = pd.read_csv(PAIKY / "test_predictions.csv")
    assert (po["id"].values == train_ids).all()
    Xp = po[list(PAIKY_5F)].to_numpy(np.float64)
    Tp = pt[list(PAIKY_5F)].to_numpy(np.float64)
    Xfm = np.column_stack([np.load(FM / f"oof_{n}.npy").astype(np.float64).reshape(-1) for n in FM_NAMES])
    Tfm = np.column_stack([np.load(FM / f"test_{n}.npy").astype(np.float64).reshape(-1) for n in FM_NAMES])
    X = np.column_stack(oof_c + [Xw, Xp, Xfm])
    Xt = np.column_stack(te_c + [Tw, Tp, Tfm])
    names = names + [f"weak_{i:02d}" for i in range(50)] + list(PAIKY_5F) + list(FM_NAMES)
    return X, Xt, names

def row_stats(P):
    mean, std = P.mean(axis=1), P.std(axis=1)
    p10, p90 = np.quantile(P, 0.10, axis=1), np.quantile(P, 0.90, axis=1)
    return np.column_stack([to_logit(mean), std, to_logit(p10), to_logit(p90)])

def group_stats(P):
    cols = []
    for lo, hi in SLICES.values():
        g = P[:, lo:hi]
        cols.append(to_logit(g.mean(axis=1)))
        cols.append(g.std(axis=1))
    return np.column_stack(cols)

def family_stats(P, names):
    rules = {
        "fam_lookup": lambda n: "lookup" in n,
        "fam_tabm": lambda n: "tabm" in n,
        "fam_lat": lambda n: n.startswith("lat") or "lattice" in n or n.startswith("latr") or n.startswith("lattri") or n.startswith("latwide") or n.startswith("latmax"),
        "fam_gbdt": lambda n: any(k in n for k in ("lgbm", "lgb", "xgb", "cat", "hgb", "et", "rf")),
        "fam_nn": lambda n: any(k in n for k in ("mlp", "realmlp", "nn2", "resnet", "tabnet", "rmlp")),
        "fam_fm": lambda n: n.startswith("fm"),
        "fam_naji": lambda n: n.startswith("naji"),
    }
    cols = []
    for _, pred in rules.items():
        idx = [i for i, n in enumerate(names) if pred(n)]
        cols.append(to_logit(P[:, idx].mean(axis=1)) if idx else np.zeros(len(P)))
    return np.column_stack(cols)

def extras(P, names):
    return np.column_stack([row_stats(P), group_stats(P), family_stats(P, names)])

def hierarchical(P, Pt, y):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    keys = list(SLICES)
    Ho = np.zeros((len(y), len(keys)))
    Ht = np.zeros((len(Pt), len(keys)))
    for i, (tr, va) in enumerate(skf.split(P, y), 1):
        fold_te = []
        for j, (lo, hi) in enumerate(SLICES.values()):
            w = fit_logreg(to_logit(P[tr, lo:hi]), y[tr], C=C)
            Ho[va, j] = predict_proba(to_logit(P[va, lo:hi]), w)
            fold_te.append(predict_proba(to_logit(Pt[:, lo:hi]), w))
        Ht += np.column_stack(fold_te) / N_FOLDS
        print(f"  [hier] fold {i} groups done", flush=True)
    return Ho, Ht

def group_lr(P, Pt, y):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    Ho = np.zeros(len(y))
    Ht = np.zeros(len(Pt))
    for tr, va in skf.split(P, y):
        w = fit_logreg(to_logit(P[tr]), y[tr], C=C)
        Ho[va] = predict_proba(to_logit(P[va]), w)
        Ht += predict_proba(to_logit(Pt), w) / N_FOLDS
    return Ho, Ht

def load_mask(n_tr, n_te):
    oofs, tes = [], []
    for name in MASK_NAMES:
        o = np.load(MASK / f"oof_{name}.npy").astype(np.float64).reshape(-1)
        t = np.load(MASK / f"test_{name}.npy").astype(np.float64).reshape(-1)
        assert o.shape[0] == n_tr and t.shape[0] == n_te
        oofs.append(o)
        tes.append(t)
    return np.column_stack(oofs), np.column_stack(tes)


## 3. 拟合元模型并写出 `submission.csv`

对照：不加 student 应约 0.970023；加上 student 应约 **0.970158**。


In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
y = train[TARGET].astype(np.int32).values
n_tr, n_te = len(train), len(test)

P, T, names = load_public(n_tr, n_te, train["id"].values)
assert P.shape[1] == 135, P.shape
Xm, Tm = load_mask(n_tr, n_te)
stu = np.clip(np.load(STU_OOF).astype(np.float64).reshape(-1), 1e-6, 1 - 1e-6)
stu_te = np.clip(np.load(STU_TE).astype(np.float64).reshape(-1), 1e-6, 1 - 1e-6)
assert stu.shape == (n_tr,) and stu_te.shape == (n_te,)
print("public", P.shape, "mask", Xm.shape, "student solo", roc_auc_score(y, stu))

Pl, Ptl = to_logit(P), to_logit(T)
Ex, Ext = extras(P, names), extras(T, names)
print("source-hier")
Ho, Ht = hierarchical(P, T, y)
print("mask-group")
Hg, Tg = group_lr(Xm, Tm, y)

X_base = np.hstack([Pl, Ex, to_logit(Ho), to_logit(Xm), to_logit(Hg)[:, None]])
Xt_base = np.hstack([Ptl, Ext, to_logit(Ht), to_logit(Tm), to_logit(Tg)[:, None]])
print("== replay mask_r3 ==")
_, _, auc0 = stack_ready(X_base, y, Xt_base, "MASK")

X = np.hstack([X_base, to_logit(stu)[:, None]])
Xt = np.hstack([Xt_base, to_logit(stu_te)[:, None]])
print("design", X.shape)
assert X.shape[1] == 169

print("== mask_student ==")
_, pred, auc = stack_ready(X, y, Xt, "STU")
print(f"STU {auc:.6f}  vs mask_r3 replay {auc0:.6f}  (recorded 0.970158)")

pred = np.clip(pred, 1e-7, 1.0 - 1e-7)
sub = pd.DataFrame({"id": test["id"], TARGET: pred})
sub.to_csv(ROOT / "submission.csv", index=False)
print("wrote submission.csv  mean", float(pred.mean()))
